In [1]:
import tensorflow as tf
import pandas as pd
import numpy as np
import os
import random
import pickle
import time

import pydicom
import matplotlib.pyplot as plt

import tensorflow.keras.backend as K
from tensorflow import keras as K
from tensorflow.keras import layers as L

from scipy.ndimage import zoom
from skimage.transform import resize
from math import ceil

2026-04-28 02:20:31.096804: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-28 02:20:31.148835: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777342831.182981      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777342831.192483      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-28 02:20:31.241640: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
raw_test = pd.read_csv('/kaggle/input/osic-pulmonary-fibrosis-progression/test.csv')
X_prediction = pd.read_csv('/kaggle/input/osic-pulmonary-fibrosis-progression/sample_submission.csv')

In [3]:
#Constant
TEST_PATH = '/kaggle/input/osic-pulmonary-fibrosis-progression/test'

#For data prep
DESIRED_SIZE = (30,256,256)
BATCH_SIZE = 256

In [4]:
# def create_submission(value):
#     if value==0:
#         sub = pd.DataFrame(data={'Patient_Week':X_prediction['Patient_Week'], 'FVC':[0 for i in X_prediction.index], 'Confidence': [10000 for i in X_prediction.index]})
#     elif value==1:
#         sub = pd.DataFrame(data={'Patient_Week':X_prediction['Patient_Week'], 'FVC':[100 for i in X_prediction.index], 'Confidence': [5000 for i in X_prediction.index]})
#     elif value==2:
#         sub = pd.DataFrame(data={'Patient_Week':X_prediction['Patient_Week'], 'FVC':[500 for i in X_prediction.index], 'Confidence': [1000 for i in X_prediction.index]})
#     elif value==3:
#         sub = pd.DataFrame(data={'Patient_Week':X_prediction['Patient_Week'], 'FVC':[1000 for i in X_prediction.index], 'Confidence': [500 for i in X_prediction.index]})
#     elif value==4:
#         sub = pd.DataFrame(data={'Patient_Week':X_prediction['Patient_Week'], 'FVC':[2000 for i in X_prediction.index], 'Confidence': [100 for i in X_prediction.index]})
#     return sub
        

# Tabular prep

In [5]:
X_prediction['Patient'] = X_prediction['Patient_Week'].str.extract(r'(.*)_.*')
X_prediction['Weeks'] = X_prediction['Patient_Week'].str.extract(r'.*_(.*)').astype(int)
X_prediction = X_prediction[['Patient', 'Weeks', 'Patient_Week']]
rename_cols = {'Weeks_y':'Base_week', 'Weeks_x': 'Weeks', 'Percent':'Base_percent', 'FVC':'Base_FVC'}
X_prediction = X_prediction.merge(raw_test, how='left', left_on='Patient', right_on='Patient').rename(columns=rename_cols)[['Patient', 'Base_week', 'Base_FVC', 'Base_percent', 'Age', 'Sex', 'SmokingStatus', 'Weeks', 'Patient_Week']].reset_index(drop=True)

In [6]:
from sklearn.preprocessing import OneHotEncoder as SklearnOneHotEncoder

#Override OneHotEncoder to have the column names created automatically (Ex-smoker, Never Smoked...) 
class OneHotEncoder(SklearnOneHotEncoder):
    def __init__(self, **kwargs):
        super(OneHotEncoder, self).__init__(**kwargs)
        self.fit_flag = False

    def fit(self, X, **kwargs):
        out = super().fit(X)
        self.fit_flag = True
        return out

    def transform(self, X, categories, index='', name='', **kwargs):
        sparse_matrix = super(OneHotEncoder, self).transform(X)
        new_columns = self.get_new_columns(X=X, name=name, categories=categories)
        d_out = pd.DataFrame(sparse_matrix.toarray(), columns=new_columns, index=index)
        return d_out

    def fit_transform(self, X, categories, index, name, **kwargs):
        self.fit(X)
        return self.transform(X, categories=categories, index=index, name=name)

    def get_new_columns(self, X, name, categories):
        new_columns = []
        for j in range(len(categories)):
            new_columns.append('{}_{}'.format(name, categories[j]))
        return new_columns
    
from sklearn.preprocessing import LabelEncoder
from sklearn.exceptions import NotFittedError

class data_preparation():
    def __init__(self):
        self.enc_sex = LabelEncoder()
        self.enc_smok = LabelEncoder()
        self.onehotenc_smok = OneHotEncoder()
        
        
    def __call__(self, data_untransformed):
        data = data_untransformed.copy(deep=True)
        
        #For the test set/Already fitted
        try:
            data['Sex'] = self.enc_sex.transform(data['Sex'].values)
            data['SmokingStatus'] = self.enc_smok.transform(data['SmokingStatus'].values)
            data = pd.concat([data.drop(columns=['SmokingStatus']), self.onehotenc_smok.transform(data['SmokingStatus'].values.reshape(-1,1), categories=self.enc_smok.classes_, name='', index=data.index).astype(int)], axis=1)
            
        #For the train set/Not yet fitted    
        except NotFittedError:
            data['Sex'] = self.enc_sex.fit_transform(data['Sex'].values)
            data['SmokingStatus'] = self.enc_smok.fit_transform(data['SmokingStatus'].values)
            data = pd.concat([data.drop(columns=['SmokingStatus']), self.onehotenc_smok.fit_transform(data['SmokingStatus'].values.reshape(-1,1), categories=self.enc_smok.classes_, name='', index=data.index).astype(int)], axis=1)
            
        return data

In [7]:
pickefile = open('/kaggle/input/prep-data/data_prep', 'rb')
data_prep = pickle.load(pickefile)
pickefile.close()

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/prep-data/data_prep'

In [8]:
X_prediction = data_prep(X_prediction).sort_values('Patient')


NameError: name 'data_prep' is not defined

# Data Generator

In [9]:
# start_time = time.perf_counter()

In [10]:
def sort_function(x): #Get the files in the right order
    return int(x.split('.')[0])

def _read(path, patients=[], desired_size=(60,512,512)):
    X = np.empty(np.concatenate(([len(patients), 1], np.array(desired_size))))
    i=0
    for patient in patients:
        list_patient_files = sorted([i for i in os.listdir(path+'/'+patient)], key=sort_function)
#         list_patient_files = list_patient_files[::max(int(len(list_patient_files)/40),1)][:30]
        df = pd.DataFrame(list_patient_files).apply(lambda x:path+'/'+patient+'/'+x)
        df = np.array(df.iloc[:,0].apply(lambda x: resize(pydicom.dcmread(x).pixel_array, DESIRED_SIZE[1:3])).to_list())
        df = zoom(df, np.array(DESIRED_SIZE)/np.array(df.shape), mode='nearest')
        X[i,0,:,:,:] = df
        i+=1
            
    #apply on resize on a batch and not on one picture
    return X

In [11]:
# X = _read('/kaggle/input/osic-pulmonary-fibrosis-progression/train', patients=['ID00007637202177411956430'], desired_size=DESIRED_SIZE)

In [12]:
temp_SELECTED_COLUMNS = ['Weeks', 'Base_week', 'Base_FVC', 'Base_percent', 'Age', 'Sex', '_Currently smokes', '_Ex-smoker', '_Never smoked']
class DataGenerator(K.utils.Sequence):
    
    def on_epoch_end(self):#Indices=np.arange(size of dataset)
        self.indices = np.arange(len(self.list_IDs))
    
    def __len__(self):
        return int(ceil(len(self.indices) / self.batch_size))
    
    def __init__(self, train, list_IDs, batch_size=1, desired_size=(10,512,512), img_path=TEST_PATH, *args, **kwargs):
        self.train = train
        self.list_IDs = list_IDs
        self.batch_size = batch_size
        self.desired_size = desired_size
        self.img_path = img_path
        self.on_epoch_end()
    
    def __getitem__(self, index):
        indices = self.indices[index*self.batch_size:(index+1)*self.batch_size]
        list_IDs_temp = [self.list_IDs[k] for k in indices]
        
        patients = self.train.loc[list_IDs_temp, 'Patient'].unique()

        imgs = _read(self.img_path, patients=patients, desired_size=self.desired_size)
        return self.train.loc[list_IDs_temp, :].reset_index(drop=True), np.transpose(imgs, (0, 2, 3, 4, 1))

In [13]:
pred_SELECTED_COLUMNS = ['Weeks', 'Base_week', 'Patient', 'Base_FVC', 'Base_percent', 'Age', 'Sex', '_Currently smokes', '_Ex-smoker', '_Never smoked']
pred_generator = DataGenerator(X_prediction[pred_SELECTED_COLUMNS], X_prediction.index, batch_size=BATCH_SIZE, desired_size=DESIRED_SIZE, img_path=TEST_PATH)

KeyError: "['_Currently smokes', '_Ex-smoker', '_Never smoked'] not in index"

# Load model and predict

In [14]:
# model = K.models.load_model('/kaggle/input/cnn-for-latent-features/model',custom_objects={'mloss':mloss(LAMBDA_LOSS)})
model = K.models.load_model('/kaggle/input/cnn-for-latent-features/model_0',compile=False)
CNN = K.models.load_model('/kaggle/input/cnn-for-latent-features/CNN_0',compile=False)

ValueError: File format not supported: filepath=/kaggle/input/cnn-for-latent-features/model_0. Keras 3 only supports V3 `.keras` files and legacy H5 format files (`.h5` extension). Note that the legacy SavedModel format is not supported by `load_model()` in Keras 3. In order to reload a TensorFlow SavedModel as an inference-only layer in Keras 3, use `keras.layers.TFSMLayer(/kaggle/input/cnn-for-latent-features/model_0, call_endpoint='serving_default')` (note that your `call_endpoint` might have a different name).

In [15]:
y_prediction = np.array([[0,0,0]])
for X1, X2 in pred_generator:
    out_imgs = CNN(X2)
    patients = X1.loc[:, 'Patient'].unique()
    X_imgs = np.empty([X1.shape[0], out_imgs.shape[1]])
    for i in range(len(patients)):
        X_imgs[np.where(X1['Patient']==patients[i])] = out_imgs.numpy()[i]
    inp_mlp = [tf.keras.layers.concatenate([X_imgs, model.get_layer('dense_to_freeze')(np.asarray(X1[temp_SELECTED_COLUMNS]))], axis=1)]
    for layer in model.layers[-3:]:
        inp_mlp.append(layer(inp_mlp[-1]))
    y_prediction = np.append(y_prediction, inp_mlp[-1].numpy(), axis=0)
y_prediction = y_prediction[1:]

NameError: name 'pred_generator' is not defined

In [16]:
sub = pd.DataFrame(data={'Patient_Week':X_prediction['Patient_Week'], 'FVC':y_prediction[:,1], 'Confidence': (y_prediction[:,2]-y_prediction[:,0])})
sub.to_csv('submission.csv', index=False)

ValueError: array length 1 does not match index length 1908